# Environment

## Imports

In [1]:
%load_ext autoreload

In [2]:
# Reload only modules imported with %aimport
%autoreload 1

# Mark these modules for auto-reload
%aimport fetch_series.core

In [3]:
import httpx
from httpx import RemoteProtocolError, ConnectError, ReadError
from httpx_retries import Retry, RetryTransport
import pandas as pd
import asyncio
import json
import re
import time
from tqdm.auto import tqdm
import fetch_series
from fetch_series.core import *
from typing import List, Tuple, Dict, Any

## Global variables

In [4]:
retry = Retry(
    total=5,
    backoff_factor=0.3,  # Wait 0.3s, 0.6s, 1.2s, 2.4s, 4.8s between retries
    status_forcelist=[
        408,  # Request Timeout - server took too long to respond
        429,  # Too Many Requests - rate limiting (common with NCBI)
        500,  # Internal Server Error - temporary server issue
        502,  # Bad Gateway - gateway/proxy error
        503,  # Service Unavailable - server overloaded/maintenance
        504,  # Gateway Timeout - gateway didn't get response in time
    ],
)

transport = RetryTransport(retry=retry)

## Helper functions

In [5]:
def eutils_link(
    dbfrom: str,
    db: str,
    ids: str | None = None,
    webenv: str | None = None,
    query_key: str | None = None,
    retmode: str = "json",
    cmd: str = "neighbor_history",
) -> Dict[str, Any]:
    """
    Search for linked items in a target database using the E-utilities API.
    Args:
        dbfrom (str): database to search from
        db (str): database to search in
        id (str): ID of the item to link
        retmode (str): return mode for the response

    Returns:
        Dict[str, Any]: JSON response from the E-utilities API containing search results
    """
    # Check that either ids or webenv and query_key are specified
    if (ids is None) == (webenv is None or query_key is None):
        raise ValueError("Must specify either ids OR webenv and query_key")

    # Retrieve linked items using the E-utilities API
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi"
    params = {
        "dbfrom": dbfrom,
        "db": db,
        "id": ids,
        "retmode": retmode,
        "WebEnv": webenv,
        "query_key": query_key,
        "cmd": cmd,
    }
    with httpx.Client(transport=transport) as client:
        response = client.get(base, params=params, timeout=10)
    response.raise_for_status()
    return response.json()

# Fetching BioProject and BioSample Metadata

## Take a look at the BioProject and BioSample databses with e-tools api

### BioProject

In [6]:
base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/einfo.fcgi"
params = {
    "db": "bioproject",
    "retmode": "json",
}

response = httpx.get(base, params=params)
response.raise_for_status()
data = response.json()
fields = data["einforesult"]["dbinfo"][0]["fieldlist"].copy()
links = data["einforesult"]["dbinfo"][0]["linklist"].copy()
del data["einforesult"]["dbinfo"][0]["fieldlist"]
del data["einforesult"]["dbinfo"][0]["linklist"]
print(json.dumps(data, indent=2))

{
  "header": {
    "type": "einfo",
    "version": "0.3"
  },
  "einforesult": {
    "dbinfo": [
      {
        "dbname": "bioproject",
        "menuname": "BioProject",
        "description": "BioProject Database",
        "dbbuild": "Build260421-0620.1",
        "count": "1038660",
        "lastupdate": "2026/04/21 07:06"
      }
    ]
  }
}


In [7]:
pd.DataFrame(fields)

,name,fullname,description,termcount,isdate,isnumerical,singletoken,hierarchy,ishidden
0,ALL,All Fields,All terms from all searchable fields,22947927,N,N,N,N,N
1,UID,UID,Unique number assigned to publication,0,N,Y,Y,N,Y
2,FILT,Filter,Limits the records,113,N,N,Y,N,N
3,ORGN,Organism,Organism,1631760,N,N,Y,Y,N
4,PRJA,Project Accession,Project Accession,1318565,N,N,Y,N,N
5,TYPE,Project Type,Project Type,2,N,N,Y,N,N
6,STPE,Project Subtype,Project Subtype,7,N,N,Y,N,N
7,DATE,Registration Date,Registration Date,8509,Y,N,Y,N,N
8,TITL,Title,Title,2569751,N,N,Y,N,N
9,CEN,Submitter Organization,Submitter Organization(s),177331,N,N,Y,N,N


In [8]:
pd.DataFrame(links)

,name,menu,description,dbto
0,bioproject_assembly_all,Assembly Links,All related Assemblies,assembly
1,bioproject_bioproject,BioProject,Links from project to related projects,bioproject
2,bioproject_bioproject_d2u,Umbrella projects,All Umbrella projects,bioproject
3,bioproject_bioproject_u2d,Data projects,All Data projects,bioproject
4,bioproject_biosample_all,BioSample Links,All related BioSamples,biosample
5,bioproject_dbvar,dbVar,Link from BioProjects to dbVar,dbvar
6,bioproject_gap,dbGaP Links,dbGaP Links,gap
7,bioproject_gds,GEO DataSet Links,GEO DataSet links,gds
8,bioproject_genome,Genome Links,Related Genomes,genome
9,bioproject_nuccore,Nucleotide Links,Related Nucleotide entry,nuccore


### BioProject

In [9]:
base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/einfo.fcgi"
params = {
    "db": "biosample",
    "retmode": "json",
}

response = httpx.get(base, params=params)
response.raise_for_status()
data = response.json()
fields = data["einforesult"]["dbinfo"][0]["fieldlist"].copy()
links = data["einforesult"]["dbinfo"][0]["linklist"].copy()
del data["einforesult"]["dbinfo"][0]["fieldlist"]
del data["einforesult"]["dbinfo"][0]["linklist"]
print(json.dumps(data, indent=2))

{
  "header": {
    "type": "einfo",
    "version": "0.3"
  },
  "einforesult": {
    "dbinfo": [
      {
        "dbname": "biosample",
        "menuname": "BioSample",
        "description": "BioSample Database",
        "dbbuild": "Build260430-0301m.1",
        "count": "54057175",
        "lastupdate": "2026/04/30 07:11"
      }
    ]
  }
}


In [10]:
pd.DataFrame(fields)

,name,fullname,description,termcount,isdate,isnumerical,singletoken,hierarchy,ishidden
0,ALL,All Fields,All terms from all searchable fields,462127089,N,N,N,N,N
1,UID,UID,Unique number assigned to publication,0,N,Y,Y,N,Y
2,FILT,Filter,Limits the records,296,N,N,Y,N,N
3,ACCN,Accession,Accession number of sequence,100585623,N,N,Y,N,N
4,TITL,Title,Words in definition line,19553227,N,N,N,N,N
5,PROP,Properties,Classification by source qualifiers and molecu...,4224,N,N,Y,N,N
6,WORD,Text Word,Free text associated with record,313550255,N,N,N,N,N
7,ORGN,Organism,"Scientific and common names of organism, and a...",1740903,N,N,Y,Y,N
8,AUTH,Author,Author(s) of publication,192502,N,N,Y,N,N
9,PDAT,Publication Date,Date sequence added to GenBank,9824,Y,N,Y,N,N


In [11]:
pd.DataFrame(links)

,name,menu,description,dbto
0,biosample_assembly,Assembly links,Assembly,assembly
1,biosample_biocollections,BioCollections,BioCollections,biocollections
2,biosample_bioproject,BioProject Links,BioProject links,bioproject
3,biosample_dbvar,dbVar Links,Links to dbVar,dbvar
4,biosample_gap,dbGaP Links,Links to dbGap Studies,gap
5,biosample_gds,GEO DataSets Links,GEO DataSets links,gds
6,biosample_nuccore,Nucleotide Links,Nucleotide links,nuccore
7,biosample_omim,OMIM links,OMIM links,omim
8,biosample_pubmed,PubMed Links,PubMed links,pubmed
9,biosample_snp,SNP Links,Related SNP record,snp


## Let's try searching for a specific BioProject

In [12]:
db = "bioproject"
search_results = eutils_search(query="PRJNA988806[PRJA]", db=db)
print(json.dumps(search_results, indent=2))

{
  "header": {
    "type": "esearch",
    "version": "0.3"
  },
  "esearchresult": {
    "count": "1",
    "retmax": "1",
    "retstart": "0",
    "querykey": "1",
    "webenv": "MCID_69f37fb422cc968577098b98",
    "idlist": [
      "988806"
    ],
    "translationset": [],
    "translationstack": [
      {
        "term": "PRJNA988806[PRJA]",
        "field": "PRJA",
        "count": "1",
        "explode": "N"
      },
      "GROUP"
    ],
    "querytranslation": "PRJNA988806[PRJA]"
  }
}


In [13]:
summary = eutils_summary(
    webenv=search_results["esearchresult"]["webenv"],
    query_key=search_results["esearchresult"]["querykey"],
    db=db
)
print(json.dumps(summary, indent=2))

{
  "header": {
    "type": "esummary",
    "version": "0.3"
  },
  "result": {
    "uids": [
      "988806"
    ],
    "988806": {
      "uid": "988806",
      "taxid": 10090,
      "project_id": 988806,
      "project_acc": "PRJNA988806",
      "project_type": "Primary submission",
      "project_data_type": "Transcriptome or Gene expression",
      "sort_by_projecttype": 317146,
      "sort_by_datatype": 293436,
      "sort_by_organism": 418525,
      "project_subtype": "",
      "project_target_scope": "Multiisolate",
      "project_target_material": "Transcriptome",
      "project_target_capture": "Whole",
      "project_methodtype": "Sequencing",
      "project_method": "",
      "project_objectives_list": [
        {
          "project_objectivestype": "Expression",
          "project_objectives": ""
        }
      ],
      "registration_date": "2023/06/28 00:00",
      "project_name": "Aspartate signaling increases the aggressiveness of lung metastases by inducing eIF5A-mediat

In [14]:
links = eutils_link(
    dbfrom="bioproject",
    db="gds",
    webenv=search_results["esearchresult"]["webenv"],
    query_key=search_results["esearchresult"]["querykey"],
)
print(json.dumps(links, indent=2))

{
  "header": {
    "type": "elink",
    "version": "0.3"
  },
  "linksets": [
    {
      "dbfrom": "bioproject",
      "ids": [
        "988806"
      ],
      "linksetdbhistories": [
        {
          "dbto": "gds",
          "linkname": "bioproject_gds",
          "querykey": "2"
        }
      ],
      "webenv": "MCID_69f37fb422cc968577098b98"
    }
  ]
}


In [15]:
summary = eutils_summary(
    webenv=links["linksets"][0]["webenv"],
    query_key=links["linksets"][0]["linksetdbhistories"][0]["querykey"],
    db="gds",
)

print(json.dumps(summary, indent=2))

{
  "header": {
    "type": "esummary",
    "version": "0.3"
  },
  "result": {
    "uids": [
      "200236084"
    ],
    "200236084": {
      "uid": "200236084",
      "accession": "GSE236084",
      "gds": "",
      "title": "Aspartate signaling increases the aggressiveness of lung metastases by inducing eIF5A-mediated translation (scRNA-Seq)",
      "summary": "Lung metastases are detected in more than half of patients with metastatic tumors. However, it remains largely unknown why the lung environment is a permissive niche for metastases. Here, we discover that pulmonary aspartate triggers a cellular signaling cascade in disseminated cancer cells resulting in a translational program that boosts lung metastasis. Specifically, we observe that patients and mice with breast cancer have high concentrations of aspartate in their lung interstitial fluid. This extracellular aspartate activates the ionotropic N-methyl-D-aspartate (NMDA) receptor in cancer cells, which induces CREB-dependen

In [16]:
links = eutils_link(
    dbfrom="bioproject",
    db="sra",
    webenv=search_results["esearchresult"]["webenv"],
    query_key=search_results["esearchresult"]["querykey"],
)
print(json.dumps(links, indent=2))

{
  "header": {
    "type": "elink",
    "version": "0.3"
  },
  "linksets": [
    {
      "dbfrom": "bioproject",
      "ids": [
        "988806"
      ],
      "linksetdbhistories": [
        {
          "dbto": "sra",
          "linkname": "bioproject_sra",
          "querykey": "3"
        },
        {
          "dbto": "sra",
          "linkname": "bioproject_sra_all",
          "querykey": "4"
        }
      ],
      "webenv": "MCID_69f37fb422cc968577098b98"
    }
  ]
}


In [17]:
summary = eutils_summary(
    webenv=links["linksets"][0]["webenv"],
    query_key=links["linksets"][0]["linksetdbhistories"][0]["querykey"],
    db="gds",
)

print(json.dumps(summary, indent=2))

{
  "header": {
    "type": "esummary",
    "version": "0.3"
  },
  "result": {
    "uids": [
      "28241758",
      "28241757",
      "28241756",
      "28241755",
      "28241754"
    ],
    "28241758": {
      "uid": "28241758",
      "expxml": "  <Summary><Title>GSM7518069: TSF2, Lungs from TSF Injection, Metastatic Colonization (d16); Mus musculus; RNA-Seq</Title><Platform instrument_model=\"Illumina NovaSeq 6000\">ILLUMINA</Platform><Statistics total_runs=\"1\" total_spots=\"332639939\" total_bases=\"39584152741\" total_size=\"13332204657\" load_done=\"true\" cluster_name=\"public\"/></Summary><Submitter acc=\"SRA1663813\" center_name=\"Laboratory of Cellular Metabolism and Metabolic Re\" contact_name=\"GEO Group\" lab_name=\"\"/><Experiment acc=\"SRX20810436\" ver=\"2\" status=\"public\" name=\"GSM7518069: TSF2, Lungs from TSF Injection, Metastatic Colonization (d16); Mus musculus; RNA-Seq\"/><Study acc=\"SRP446371\" name=\"Aspartate signaling increases the aggressiveness of lu

In [18]:
summary = eutils_summary(
    webenv=links["linksets"][0]["webenv"],
    query_key=links["linksets"][0]["linksetdbhistories"][1]["querykey"],
    db="gds",
)

print(json.dumps(summary, indent=2))

{
  "header": {
    "type": "esummary",
    "version": "0.3"
  },
  "result": {
    "uids": [
      "28241758",
      "28241757",
      "28241756",
      "28241755",
      "28241754"
    ],
    "28241758": {
      "uid": "28241758",
      "expxml": "  <Summary><Title>GSM7518069: TSF2, Lungs from TSF Injection, Metastatic Colonization (d16); Mus musculus; RNA-Seq</Title><Platform instrument_model=\"Illumina NovaSeq 6000\">ILLUMINA</Platform><Statistics total_runs=\"1\" total_spots=\"332639939\" total_bases=\"39584152741\" total_size=\"13332204657\" load_done=\"true\" cluster_name=\"public\"/></Summary><Submitter acc=\"SRA1663813\" center_name=\"Laboratory of Cellular Metabolism and Metabolic Re\" contact_name=\"GEO Group\" lab_name=\"\"/><Experiment acc=\"SRX20810436\" ver=\"2\" status=\"public\" name=\"GSM7518069: TSF2, Lungs from TSF Injection, Metastatic Colonization (d16); Mus musculus; RNA-Seq\"/><Study acc=\"SRP446371\" name=\"Aspartate signaling increases the aggressiveness of lu

In [19]:
links = eutils_link(
    dbfrom="bioproject",
    db="biosample",
    webenv=search_results["esearchresult"]["webenv"],
    query_key=search_results["esearchresult"]["querykey"],
)
print(json.dumps(links, indent=2))

{
  "header": {
    "type": "elink",
    "version": "0.3"
  },
  "linksets": [
    {
      "dbfrom": "bioproject",
      "ids": [
        "988806"
      ],
      "linksetdbhistories": [
        {
          "dbto": "biosample",
          "linkname": "bioproject_biosample",
          "querykey": "5"
        },
        {
          "dbto": "biosample",
          "linkname": "bioproject_biosample_all",
          "querykey": "6"
        }
      ],
      "webenv": "MCID_69f37fb422cc968577098b98"
    }
  ]
}


In [20]:
summary = eutils_summary(
    webenv=links["linksets"][0]["webenv"],
    query_key=links["linksets"][0]["linksetdbhistories"][0]["querykey"],
    db="gds",
)

print(json.dumps(summary, indent=2))

{
  "header": {
    "type": "esummary",
    "version": "0.3"
  },
  "result": {
    "uids": [
      "36028301",
      "36028300",
      "36028299",
      "36028298",
      "36028297"
    ],
    "36028301": {
      "uid": "36028301",
      "title": "CM1, Lungs from CM Injection, Metastatic Seeding (d11)",
      "accession": "SAMN36028301",
      "date": "2024/10/04",
      "publicationdate": "2024/10/04",
      "modificationdate": "2024/10/04",
      "organization": "Laboratory of Cellular Metabolism and Metabolic Regulation, VIB-KU Leuven Center for Cancer Biology, VIB/KU Leuven",
      "taxonomy": "10090",
      "organism": "Mus musculus",
      "sourcesample": "BioSample:SAMN36028301",
      "sampledata": "<BioSample access=\"public\" publication_date=\"2024-10-04T00:00:00.000\" last_update=\"2024-10-04T14:36:16.933\" submission_date=\"2023-06-28T20:00:08.023\" id=\"36028301\" accession=\"SAMN36028301\">   <Ids>     <Id db=\"BioSample\" is_primary=\"1\">SAMN36028301</Id>     <Id db=\